## Install Dependencies

In [1]:
!uv pip install syftbox-enclave syft-rds==0.1.1-dev.4

Resolved 65 packages in 848ms                                        
Prepared 1 package in 1.63s                                              
Installed 38 packages in 183ms                              
 + annotated-types==0.7.0
 + bracex==2.6
 + click==8.2.1
 + contourpy==1.3.2
 + cryptography==45.0.5
 + cycler==0.12.1
 + dnspython==2.7.0
 + email-validator==2.2.0
 + fonttools==4.59.0
 + ipywidgets==8.1.7
 + jupyterlab-widgets==3.0.15
 + kiwisolver==1.4.8
 + loguru==0.7.3
 + markdown-it-py==3.0.0
 + matplotlib==3.10.3
 + mdurl==0.1.2
 + nh3==0.3.0
 + numpy==2.3.1
 + pandas==2.3.1
 + pathspec==0.12.1
 + pillow==11.3.0
 + pydantic==2.11.7
 + pydantic-core==2.33.2
 + pyparsing==3.2.3
 + pytz==2025.2
 + rich==14.0.0
 + shellingham==1.5.4
 + syft-core==0.2.8
 + syft-event==0.2.8
 + syft-rds==0.1.1.dev4
 + syft-rpc==0.2.8
 + syftbox-enclave==0.1.4
 + typer==0.16.0
 + typing-inspection==0.4.1
 + tzdata==2025.2
 + watchdog==6.0.0
 + wcmatch==10.0
 + widgetsnbextension==4.0.14


In [2]:
from syft_rds import init_session
from syft_core import Client

In [3]:
DATA_OWNERS = [
            # "do-1-ndl@openmined.org",
            # "do-2-ndl@openmined.org",
            # "<DO-1-EMAIL>"
              ]
DATA_SCIENTIST = [ Client.load().email ]

In [4]:
ds_clients = []

for do in DATA_OWNERS:
    ds_client = init_session(do)
    ds_clients.append(ds_client)
    print(f"Logged into {ds_client.host}")


Logged into rasswanth@openmined.org


In [5]:
datasets = []

# We could either index the datasets
for ds_client in ds_clients:
    datasets.append(ds_client.datasets[0])
    
    

In [6]:
datasets

[Dataset
   uid: 324c8d42-8b3c-4172-923a-5e23ef313b5d
   created_by: None
   created_at: 2025-07-18T13:51:57.422657Z
   updated_at: 2025-07-18T13:52:10.820391Z
   client_id: aba1c811-528a-4b79-827e-30791f004b34
   name: Organic Crop Data
   private: syft://rasswanth@openmined.org/private/datasets/Organic Crop Data
   mock: syft://rasswanth@openmined.org/public/datasets/Organic Crop Data
   summary: 
   readme: syft://rasswanth@openmined.org/public/datasets/Organic Crop Data/dummy_description.txt
   tags: []
   runtime: {'cmd': ['python'], 'image_name': None, 'mount_dir': None}
   auto_approval: ['rasswanth@openmined.org']]

### Inspect the data and inform their analysis



In [8]:
# Inspect the mock data of all datasets

import pandas as pd
from pathlib import Path
from IPython.display import display

for dataset in datasets:
    mock_path = dataset.get_mock_path()
    mock_csv_path = list(Path(mock_path).glob("*.csv"))[0]
    df_mock = pd.read_csv(mock_csv_path)
    
    display(df_mock.head(3))
    print("\n\n\n")
    

,ID,Product name,Quantity,Price ($),Unit
0,U1399,Radishes,121,410.75,kgs
1,U9734,Kale,450,1194.47,kgs
2,U4696,Spinach,88,143.86,kgs


### Propose analysis

In [12]:

code_path = Path(".") / "code"
code_path.mkdir(exist_ok=True)

code_file_name  = "crop_analysis.py"
code_file_path = code_path / code_file_name

In [15]:
%%writefile {code_file_path}

import os
from pathlib import Path
from sys import exit

import pandas as pd

DATA_DIR = os.environ["DATA_DIR"]
OUTPUT_DIR = os.environ["OUTPUT_DIR"]

dataset_paths = [ Path(dataset_path) for dataset_path in DATA_DIR.split(",")]
csv_paths = []
for dataset_path in dataset_paths:
    csv_paths.extend(list(Path(dataset_path).glob("*.csv")))

total_carrots = 0
total_tomatoes = 0

for csv_path in csv_paths:
    if not csv_path.exists():
        print(f"Warning: CSV path does not exist: {csv_path}")
        exit(1)
    df = pd.read_csv(csv_path)
    total_carrots += df[df["Product name"] == "Carrots"]["Quantity"].sum()
    total_tomatoes += df[df["Product name"] == "Tomatoes"]["Quantity"].sum()

print(f"Total Carrots: {total_carrots}\n")
print(f"Total Tomatoes: {total_tomatoes}\n")

with open(os.path.join(OUTPUT_DIR, "output.txt"), "w") as f:
    f.write(f"Total Carrots: {total_carrots}\n")
    f.write(f"Total Tomatoes: {total_tomatoes}\n")

Overwriting code/crop_analysis.py


### Test the analysis code against mock data

Before submitting the code for review, Treasury can test their analysis against the Department of Health mock data to ensure it works correctly.


In [16]:
# Test against mock of all datasets

import subprocess, tempfile, os

with tempfile.TemporaryDirectory() as temp_dir:
    env = os.environ.copy()
    env.update({'DATA_DIR': ",".join([ str(dataset.get_mock_path()) for dataset in datasets]), 'OUTPUT_DIR': temp_dir})
    
    result = subprocess.run(["python", str(code_file_path)], env=env, capture_output=True, text=True)
    print(result.stdout)

Total Carrots: 0

Total Tomatoes: 89




In [17]:
from uuid import uuid4

# Generate 
RANDOM_ID = str(uuid4())[0:8]
JOB_NAME = f"Test Job - {RANDOM_ID}"

print("Job Name:", JOB_NAME)

Job Name: Test Job - e1385b71


In [18]:
ENCLAVE = "enclave-organic-coop@openmined.org"

jobs = []

for ds_client, dataset in zip(ds_clients, datasets):
    job = ds_client.jobs.submit(
                name=JOB_NAME,
                description="Organic Coop Experiment",
                user_code_path=code_path,
                dataset_name=dataset.name,
                entrypoint = code_file_name,
                enclave = ENCLAVE
            )
    job.describe()
    jobs.append(job)

uid,ed8ff141-7898-4279-a2cc-ae69fff8840d
created_by,rasswanth@openmined.org
created_at,2025-07-18 14:04:25
updated_at,2025-07-18 14:04:25
name,Test Job - e1385b71
description,Organic Coop Experiment
status,pending_code_review
error,no_error
error_message,None
output_path,/Users/rasswanths/SyftBox/datasites/rasswanth@openmined.org/app_data/RDS/user_files/rasswanth@openmined.org/Job/ed8ff141-7898-4279-a2cc-ae69fff8840d
dataset_name,Organic Crop Data


In [19]:
ds_clients[0].jobs.get(name=JOB_NAME).describe()

uid,ed8ff141-7898-4279-a2cc-ae69fff8840d
created_by,rasswanth@openmined.org
created_at,2025-07-18 14:04:25
updated_at,2025-07-18 14:04:25
name,Test Job - e1385b71
description,Organic Coop Experiment
status,approved
error,no_error
error_message,None
output_path,/Users/rasswanths/SyftBox/datasites/rasswanth@openmined.org/app_data/RDS/user_files/rasswanth@openmined.org/Job/ed8ff141-7898-4279-a2cc-ae69fff8840d
dataset_name,Organic Crop Data


## Enclave Client

In [20]:
from syftbox_enclave import connect

In [21]:
enclave_client = connect(ENCLAVE)

In [22]:
PROJECT_NAME = f"Test Project - {RANDOM_ID}"

print("Project Name:", PROJECT_NAME)

Project Name: Test Project - e1385b71


In [23]:
datasets

[Dataset
   uid: 324c8d42-8b3c-4172-923a-5e23ef313b5d
   created_by: None
   created_at: 2025-07-18T13:51:57.422657Z
   updated_at: 2025-07-18T13:52:10.820391Z
   client_id: aba1c811-528a-4b79-827e-30791f004b34
   name: Organic Crop Data
   private: syft://rasswanth@openmined.org/private/datasets/Organic Crop Data
   mock: syft://rasswanth@openmined.org/public/datasets/Organic Crop Data
   summary: 
   readme: syft://rasswanth@openmined.org/public/datasets/Organic Crop Data/dummy_description.txt
   tags: []
   runtime: {'cmd': ['python'], 'image_name': None, 'mount_dir': None}
   auto_approval: ['rasswanth@openmined.org']]

In [24]:

proj_res = enclave_client.create_project(
                   project_name = PROJECT_NAME,
                   datasets = datasets,
                   output_owners = DATA_OWNERS + DATA_SCIENTIST,
                   code_path = code_path,
                   entrypoint = code_file_name,
            )

2025-07-18 19:34:52.557 | INFO     | syftbox_enclave.client:create_project:100 - Project Test Project - e1385b71 created in enclave app path /Users/rasswanths/SyftBox/datasites/enclave-organic-coop@openmined.org/app_data/enclave.


In [25]:
proj_res.status(block=True)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Project: Test Project - e1385b71                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Dataset ID                           ┃ Host                    ┃ Status     ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ 324c8d42-8b3c-4172-923a-5e23ef313b5d │ rasswanth@openmined.org │ pending 🟠 │
└──────────────────────────────────────┴─────────────────────────┴────────────┘

2025-07-18 19:35:09.063 | WARNING  | syftbox_enclave.client:_get_metrics:130 - Metrics file does not exist for 
 project: {self.project_name}.
 The project might have completed 
 Kindly call .output() to check if the output is available.


All datasets have uploaded to the Enclave! ✅

'All success.'

In [26]:
# Force Start if atleast one of the datasites have approved
# proj_res.force_start()


# 5. Access Output Results

In [27]:
# Wait until the project is ready
proj_res_path = proj_res.output(block=True)

2025-07-18 19:36:18.858 | INFO     | syftbox_enclave.client:output:231 - Waiting for output for project Test Project - e1385b71...
2025-07-18 19:36:18.859 | INFO     | syftbox_enclave.client:output:236 - Output available for project Test Project - e1385b71 ✅
 Directory: /Users/rasswanths/SyftBox/datasites/enclave-organic-coop@openmined.org/app_data/enclave/jobs/outputs/Test Project - e1385b71.


In [29]:
output_file_path = proj_res_path / "output.txt"

In [31]:
with open(output_file_path ,"r") as f:
    print(f.read())


Total Carrots: 345
Total Tomatoes: 0

